# Step 2 — Model Comparison (Validation Only)

**Purpose:** Compare candidate fraud-detection models using the **validation set only**.

### Data policy
- **Train:** used to fit models.
- **Validation:** used to compare models.
- **Final Test:** created and kept completely untouched in this notebook.
- No threshold optimization is performed here.
- No model is saved here.

### Models compared
1. Logistic Regression + Class Weight
2. Logistic Regression + SMOTE
3. XGBoost

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

In [4]:
data_path = Path("../data/training/creditcard.csv")
df = pd.read_csv(data_path)

df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [5]:
print("Dataset shape:", df.shape)

Dataset shape: (284807, 31)


In [6]:
X = df.drop("Class", axis=1)
y = df["Class"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (284807, 30)
Target: (284807,)


In [7]:
# 20% is reserved as the final held-out test set.
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Split the remaining 80% into 60% train and 20% validation.
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=42
)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("\nClass distribution:")
print("Train:")
print(y_train.value_counts())
print("\nValidation:")
print(y_val.value_counts())
print("\nFinal Test (untouched in Step 3):")
print(y_test.value_counts())

X_train: (170883, 30)
X_val  : (56962, 30)
X_test : (56962, 30)

Class distribution:
Train:
Class
0    170588
1       295
Name: count, dtype: int64

Validation:
Class
0    56863
1       99
Name: count, dtype: int64

Final Test (untouched in Step 3):
Class
0    56864
1       98
Name: count, dtype: int64


In [8]:
# Feature Scaling
# Scaling is needed for Logistic Regression.
# Fit ONLY on training data.

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("Training scaled shape  :", X_train_scaled.shape)
print("Validation scaled shape:", X_val_scaled.shape)

Training scaled shape  : (170883, 30)
Validation scaled shape: (56962, 30)


### Model 1 — Logistic Regression + Class Weight

In [9]:
lr_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train_scaled, y_train)
print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [ ]:
lr_probability = lr_model.predict_proba(X_val_scaled)[:, 1]
lr_prediction = (lr_probability >= 0.50).astype(int)
print("Validation predictions generated:", len(lr_probability))

Validation predictions generated: 56962


In [ ]:
lr_precision = precision_score(y_val, lr_prediction, zero_division=0)
lr_recall = recall_score(y_val, lr_prediction, zero_division=0)
lr_f1 = f1_score(y_val, lr_prediction, zero_division=0)
lr_pr_auc = average_precision_score(y_val, lr_probability)
lr_roc_auc = roc_auc_score(y_val, lr_probability)

print("=" * 55)
print("LOGISTIC REGRESSION — VALIDATION")
print("=" * 55)
print(f"Precision: {lr_precision:.4f}")
print(f"Recall:    {lr_recall:.4f}")
print(f"F1 Score:  {lr_f1:.4f}")
print(f"PR-AUC:    {lr_pr_auc:.4f}")
print(f"ROC-AUC:   {lr_roc_auc:.4f}")

LOGISTIC REGRESSION — VALIDATION
Precision: 0.0597
Recall:    0.8990
F1 Score:  0.1119
PR-AUC:    0.6807
ROC-AUC:   0.9747


### Model 2 — Logistic Regression + SMOTE

SMOTE is applied **only to the training data**. Validation data keeps its original distribution.

In [9]:
# ---------------------------------
# Step 9: SMOTE on Training Data Only
# ---------------------------------

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:")
print(y_train.value_counts())
print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

Before SMOTE:
Class
0    170588
1       295
Name: count, dtype: int64

After SMOTE:
Class
0    170588
1    170588
Name: count, dtype: int64


In [ ]:
# Logistic Regression + SMOTE

lr_smote_model = LogisticRegression(max_iter=1000, random_state=42)
lr_smote_model.fit(X_train_smote, y_train_smote)
print("Logistic Regression + SMOTE trained successfully.")

Logistic Regression + SMOTE trained successfully.


In [ ]:
# SMOTE Model Validation Prediction

lr_smote_probability = lr_smote_model.predict_proba(X_val_scaled)[:, 1]
lr_smote_prediction = (lr_smote_probability >= 0.50).astype(int)
print("Validation predictions generated:", len(lr_smote_probability))

Validation predictions generated: 56962


In [ ]:
# SMOTE Model Validation Metrics

lr_smote_precision = precision_score(y_val, lr_smote_prediction, zero_division=0)
lr_smote_recall = recall_score(y_val, lr_smote_prediction, zero_division=0)
lr_smote_f1 = f1_score(y_val, lr_smote_prediction, zero_division=0)
lr_smote_pr_auc = average_precision_score(y_val, lr_smote_probability)
lr_smote_roc_auc = roc_auc_score(y_val, lr_smote_probability)

print("=" * 55)
print("LOGISTIC REGRESSION + SMOTE — VALIDATION")
print("=" * 55)
print(f"Precision: {lr_smote_precision:.4f}")
print(f"Recall:    {lr_smote_recall:.4f}")
print(f"F1 Score:  {lr_smote_f1:.4f}")
print(f"PR-AUC:    {lr_smote_pr_auc:.4f}")
print(f"ROC-AUC:   {lr_smote_roc_auc:.4f}")

LOGISTIC REGRESSION + SMOTE — VALIDATION
Precision: 0.0559
Recall:    0.8788
F1 Score:  0.1051
PR-AUC:    0.6750
ROC-AUC:   0.9718


### Model 3 — XGBoost

XGBoost does not need feature scaling here. `scale_pos_weight` is calculated from the training set only.

In [ ]:
# XGBoost Class Imbalance

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
scale_pos_weight = negative_count / positive_count

print("Negative samples:", negative_count)
print("Positive samples:", positive_count)
print("scale_pos_weight:", scale_pos_weight)

Negative samples: 170588
Positive samples: 295
scale_pos_weight: 578.264406779661


In [ ]:
# Train XGBoost

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)
print("XGBoost trained successfully.")

XGBoost trained successfully.


In [ ]:
# XGBoost Validation Prediction

xgb_probability = xgb_model.predict_proba(X_val)[:, 1]
xgb_prediction = (xgb_probability >= 0.50).astype(int)
print("Validation predictions generated:", len(xgb_probability))

Validation predictions generated: 56962


In [ ]:
# XGBoost Validation Metrics

xgb_precision = precision_score(y_val, xgb_prediction, zero_division=0)
xgb_recall = recall_score(y_val, xgb_prediction, zero_division=0)
xgb_f1 = f1_score(y_val, xgb_prediction, zero_division=0)
xgb_pr_auc = average_precision_score(y_val, xgb_probability)
xgb_roc_auc = roc_auc_score(y_val, xgb_probability)

print("=" * 55)
print("XGBOOST — VALIDATION")
print("=" * 55)
print(f"Precision: {xgb_precision:.4f}")
print(f"Recall:    {xgb_recall:.4f}")
print(f"F1 Score:  {xgb_f1:.4f}")
print(f"PR-AUC:    {xgb_pr_auc:.4f}")
print(f"ROC-AUC:   {xgb_roc_auc:.4f}")

XGBOOST — VALIDATION
Precision: 0.8953
Recall:    0.7778
F1 Score:  0.8324
PR-AUC:    0.8276
ROC-AUC:   0.9766


## Step 17 — Model Comparison

All models are compared on the same validation set at threshold 0.50. The final test set is not used to select the model.

In [ ]:
#  Model Comparison

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression + Class Weight",
        "Logistic Regression + SMOTE",
        "XGBoost"
    ],
    "Precision": [lr_precision, lr_smote_precision, xgb_precision],
    "Recall": [lr_recall, lr_smote_recall, xgb_recall],
    "F1": [lr_f1, lr_smote_f1, xgb_f1],
    "PR-AUC": [lr_pr_auc, lr_smote_pr_auc, xgb_pr_auc],
    "ROC-AUC": [lr_roc_auc, lr_smote_roc_auc, xgb_roc_auc]
})

comparison = comparison.sort_values("PR-AUC", ascending=False).reset_index(drop=True)
print(comparison.to_string(index=False))

                             Model  Precision   Recall       F1   PR-AUC  ROC-AUC
                           XGBoost   0.895349 0.777778 0.832432 0.827634 0.976569
Logistic Regression + Class Weight   0.059651 0.898990 0.111879 0.680711 0.974728
       Logistic Regression + SMOTE   0.055913 0.878788 0.105136 0.675016 0.971811
